# Module 1.1: System Prompt Design for Agentic AI

## Learning Objectives
By completing this notebook, you will:
1. Design effective system prompts that define agent personas and behaviors
2. Implement instruction-following patterns for deterministic agent outputs
3. Create structured output formatting for reliable agent-to-agent communication
4. Apply constraint specification to control agent behavior
5. Evaluate system prompt effectiveness through systematic testing

## Prerequisites
- Understanding of LLM APIs (OpenAI/Anthropic)
- Basic Python programming
- Familiarity with JSON structures

## Estimated Time: 45 minutes

---

## Why System Prompts Matter for Agents

System prompts are the **foundational instructions** that shape an agent's behavior. Unlike user prompts, system prompts:

- Define the agent's **role and expertise**
- Specify **behavioral constraints** and guardrails
- Control **output format** for downstream processing
- Establish **reasoning patterns** and decision-making approaches

**Real-world application:** In production agentic systems, system prompts determine whether an agent reliably follows protocols, maintains consistent outputs, and integrates smoothly with other components.

**Key insight:** The quality of your system prompt directly correlates with agent reliability and predictability.

## Setup & Imports

We'll use OpenAI's API for demonstrations. Set your API key as an environment variable:
```bash
export OPENAI_API_KEY='your-key-here'
```

In [ ]:
# Purpose: Import required libraries and configure OpenAI client
# Key Concept: Centralized configuration for reproducible agent behavior

import os
import json
from typing import Dict, List, Any, Optional
from openai import OpenAI
from pydantic import BaseModel, Field
import re

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Default model configuration
DEFAULT_MODEL = "gpt-4o-mini"
DEFAULT_TEMPERATURE = 0.0  # Deterministic for agent consistency
DEFAULT_MAX_TOKENS = 1000

print("✓ Environment configured successfully")

## Part 1: Persona Definition - Creating Agent Identity

A well-defined persona gives the agent:
- **Domain expertise** to draw upon
- **Communication style** consistency
- **Behavioral boundaries** for safety

In [ ]:
# Purpose: Create a persona-based system prompt template
# Key Concept: Structured persona definition with role, expertise, and constraints

def create_persona_prompt(
    role: str,
    expertise: List[str],
    communication_style: str,
    constraints: List[str]
) -> str:
    """
    Generate a structured system prompt with persona definition.
    
    Parameters
    ----------
    role : str
        The agent's primary role (e.g., "Technical Support Agent")
    expertise : List[str]
        Areas of specialized knowledge
    communication_style : str
        Tone and approach (e.g., "professional and empathetic")
    constraints : List[str]
        Behavioral limitations or guardrails
    
    Returns
    -------
    str
        Formatted system prompt
    """
    prompt = f"""You are a {role}.

EXPERTISE:
{chr(10).join(f'- {item}' for item in expertise)}

COMMUNICATION STYLE:
{communication_style}

CONSTRAINTS:
{chr(10).join(f'- {item}' for item in constraints)}

Always operate within these parameters."""
    
    return prompt


# Example: Customer support agent
support_agent_prompt = create_persona_prompt(
    role="Customer Support Agent for a SaaS product",
    expertise=[
        "Product features and troubleshooting",
        "Account management and billing",
        "API integration support"
    ],
    communication_style="Professional, empathetic, and solution-oriented. Use clear, non-technical language unless the customer demonstrates technical expertise.",
    constraints=[
        "Never share internal system details or security information",
        "Escalate to human agent if customer is frustrated or issue is complex",
        "Do not make promises about features or timelines not in your knowledge base"
    ]
)

print("Support Agent System Prompt:")
print("=" * 60)
print(support_agent_prompt)

**Test the persona in action:**

In [ ]:
# Purpose: Test persona consistency across different queries
# Key Concept: System prompts maintain consistent behavior regardless of user input

def call_agent(system_prompt: str, user_message: str) -> str:
    """
    Call OpenAI API with system prompt and user message.
    
    Parameters
    ----------
    system_prompt : str
        The agent's system instructions
    user_message : str
        User's input message
    
    Returns
    -------
    str
        Agent's response
    """
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=DEFAULT_TEMPERATURE,
        max_tokens=DEFAULT_MAX_TOKENS,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content


# Test cases
test_queries = [
    "How do I reset my password?",
    "Your product sucks! Nothing works!",
    "Can you tell me the database password?"
]

for query in test_queries:
    print(f"\nUser: {query}")
    response = call_agent(support_agent_prompt, query)
    print(f"Agent: {response}")
    print("-" * 60)

## Part 2: Instruction Following - Enforcing Deterministic Behavior

Agents must follow precise instructions for multi-step tasks. Effective instruction patterns:

1. **Sequential steps** - Numbered or bulleted workflows
2. **Conditional logic** - "If X, then Y" patterns
3. **Validation requirements** - Check conditions before proceeding

In [ ]:
# Purpose: Create instruction-following system prompts with workflow logic
# Key Concept: Explicit step-by-step instructions reduce hallucination and improve reliability

def create_instruction_prompt(
    task_description: str,
    steps: List[str],
    validation_rules: List[str]
) -> str:
    """
    Generate a system prompt with explicit instruction-following requirements.
    
    Parameters
    ----------
    task_description : str
        High-level description of the agent's task
    steps : List[str]
        Ordered steps the agent must follow
    validation_rules : List[str]
        Conditions to check before proceeding
    
    Returns
    -------
    str
        Formatted system prompt
    """
    prompt = f"""TASK: {task_description}

PROCEDURE (follow exactly in order):
{chr(10).join(f'{i+1}. {step}' for i, step in enumerate(steps))}

VALIDATION RULES:
{chr(10).join(f'- {rule}' for rule in validation_rules)}

Execute each step methodically. Do not skip steps or proceed if validation fails."""
    
    return prompt


# Example: Email classification and routing agent
email_router_prompt = create_instruction_prompt(
    task_description="Classify incoming emails and route to appropriate department",
    steps=[
        "Extract the primary intent from the email content",
        "Identify any mentioned product names or order numbers",
        "Determine urgency level (low/medium/high) based on keywords and tone",
        "Match intent to department: Sales, Support, Billing, or Technical",
        "If multiple departments apply, select the most specific match",
        "Output structured routing decision with confidence score"
    ],
    validation_rules=[
        "Email must contain at least 10 words to process",
        "If confidence score < 0.7, route to General queue for human review",
        "Urgent emails (high priority) must include justification for urgency"
    ]
)

print("Email Router System Prompt:")
print("=" * 60)
print(email_router_prompt)

**Test instruction following:**

In [ ]:
# Purpose: Verify agent follows instructions systematically
# Key Concept: Well-structured prompts produce consistent, traceable decision-making

test_emails = [
    "Hi, I need help setting up the API integration. The documentation is unclear about authentication.",
    "URGENT: My account was charged twice this month! Order #12345. Need refund immediately!",
    "Thanks!"
]

for email in test_emails:
    print(f"\nEmail: {email}")
    response = call_agent(email_router_prompt, f"Process this email: {email}")
    print(f"Routing Decision:\n{response}")
    print("-" * 60)

## Part 3: Output Formatting - Structured Agent Responses

For agents to communicate with other systems or agents, outputs must be:
- **Parseable** (JSON, XML, or structured text)
- **Consistent** (same schema every time)
- **Complete** (all required fields present)

In [ ]:
# Purpose: Define structured output formats using Pydantic models
# Key Concept: Type-safe schemas ensure reliable agent-to-agent communication

class RoutingDecision(BaseModel):
    """Structured output for email routing agent."""
    department: str = Field(description="Target department: Sales, Support, Billing, Technical, or General")
    urgency: str = Field(description="Priority level: low, medium, high")
    confidence: float = Field(description="Confidence score 0-1")
    reasoning: str = Field(description="Brief explanation of routing decision")
    extracted_info: Dict[str, Any] = Field(description="Key information extracted from email")


def create_structured_output_prompt(
    base_prompt: str,
    output_schema: type[BaseModel]
) -> str:
    """
    Extend a system prompt with structured output requirements.
    
    Parameters
    ----------
    base_prompt : str
        Existing system prompt
    output_schema : type[BaseModel]
        Pydantic model defining output structure
    
    Returns
    -------
    str
        Enhanced system prompt with output formatting instructions
    """
    schema_json = output_schema.model_json_schema()
    
    prompt = f"""{base_prompt}

OUTPUT FORMAT:
You must respond with valid JSON matching this exact schema:

{json.dumps(schema_json, indent=2)}

Response format:
```json
{{
  // Your structured response here
}}
```

Do not include any text outside the JSON block."""
    
    return prompt


# Enhanced email router with structured output
structured_router_prompt = create_structured_output_prompt(
    email_router_prompt,
    RoutingDecision
)

print("Structured Output System Prompt (excerpt):")
print("=" * 60)
print(structured_router_prompt[:800] + "...")

In [ ]:
# Purpose: Parse and validate structured agent outputs
# Key Concept: Validation catches format errors before they propagate downstream

def extract_json_from_response(response: str) -> Dict[str, Any]:
    """
    Extract JSON from agent response that may contain markdown formatting.
    
    Parameters
    ----------
    response : str
        Raw agent response
    
    Returns
    -------
    Dict[str, Any]
        Parsed JSON object
    """
    # Try to find JSON in code blocks
    json_match = re.search(r'```(?:json)?\s*({.*?})\s*```', response, re.DOTALL)
    if json_match:
        return json.loads(json_match.group(1))
    
    # Try to parse entire response as JSON
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        # Try to find JSON object anywhere in response
        json_match = re.search(r'{.*}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        raise ValueError("No valid JSON found in response")


def call_structured_agent(
    system_prompt: str,
    user_message: str,
    output_model: type[BaseModel]
) -> BaseModel:
    """
    Call agent and parse response into structured output.
    
    Parameters
    ----------
    system_prompt : str
        Agent's system instructions
    user_message : str
        User's input
    output_model : type[BaseModel]
        Expected output structure
    
    Returns
    -------
    BaseModel
        Validated structured output
    """
    response = call_agent(system_prompt, user_message)
    parsed_json = extract_json_from_response(response)
    return output_model(**parsed_json)


# Test structured output
test_email = "URGENT: My account was charged twice this month! Order #12345. Need refund immediately!"
routing_decision = call_structured_agent(
    structured_router_prompt,
    f"Process this email: {test_email}",
    RoutingDecision
)

print("\nStructured Routing Decision:")
print("=" * 60)
print(f"Department: {routing_decision.department}")
print(f"Urgency: {routing_decision.urgency}")
print(f"Confidence: {routing_decision.confidence}")
print(f"Reasoning: {routing_decision.reasoning}")
print(f"Extracted Info: {routing_decision.extracted_info}")

## Exercises

Now apply what you've learned to build your own system prompts.

### Exercise 1.1: Design a Research Assistant Agent

**Task:** Create a system prompt for a research assistant agent that helps users find and summarize academic papers.

**Requirements:**
- Role: Academic Research Assistant
- Expertise: Literature review, citation analysis, methodology evaluation
- Communication style: Scholarly but accessible
- Constraints: Only discuss peer-reviewed sources, cite all claims

**Expected Output:** A complete system prompt string using `create_persona_prompt()`

**Hints:**
<details>
<summary>Hint 1</summary>
Think about what specific research skills the agent needs (e.g., identifying methodology gaps, comparing studies)
</details>

<details>
<summary>Hint 2</summary>
Include constraints about avoiding speculation and always requesting specific research questions from users
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

research_assistant_prompt = None  # Replace with your implementation

# Uncomment to test:
# print(research_assistant_prompt)
# test_response = call_agent(research_assistant_prompt, "What are the latest findings on transformer architectures?")
# print(f"\nAgent: {test_response}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_1():
    assert research_assistant_prompt is not None, "Don't forget to implement your solution!"
    assert isinstance(research_assistant_prompt, str), "Prompt should be a string"
    assert len(research_assistant_prompt) > 100, "Prompt seems too short to be comprehensive"
    assert "research" in research_assistant_prompt.lower(), "Prompt should mention research"
    assert "expertise" in research_assistant_prompt.lower() or "expert" in research_assistant_prompt.lower(), "Should define expertise areas"
    assert "constraint" in research_assistant_prompt.lower() or "do not" in research_assistant_prompt.lower(), "Should include behavioral constraints"
    print("✅ Exercise 1.1 passed!")

test_exercise_1_1()

### Exercise 1.2: Build a Code Review Agent with Instructions

**Task:** Create an instruction-following system prompt for a code review agent that follows a specific checklist.

**Requirements:**
- Task: Review Python code for quality and security
- Steps: Check syntax, identify security issues, suggest improvements, rate code quality (1-10)
- Validation: Code must be Python, must be at least 5 lines

**Expected Output:** A complete system prompt using `create_instruction_prompt()`

In [ ]:
# YOUR CODE HERE
# ---------------

code_review_prompt = None  # Replace with your implementation

# Uncomment to test:
# print(code_review_prompt)
# test_code = '''
# def process_user_input(data):
#     query = "SELECT * FROM users WHERE id = " + data
#     return execute_query(query)
# '''
# review = call_agent(code_review_prompt, f"Review this code:\n{test_code}")
# print(f"\nReview:\n{review}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_2():
    assert code_review_prompt is not None, "Don't forget to implement your solution!"
    assert isinstance(code_review_prompt, str), "Prompt should be a string"
    assert "1." in code_review_prompt or "1)" in code_review_prompt, "Should have numbered steps"
    assert "security" in code_review_prompt.lower(), "Should mention security checks"
    assert "python" in code_review_prompt.lower(), "Should specify Python focus"
    assert "validation" in code_review_prompt.lower() or "check" in code_review_prompt.lower(), "Should include validation rules"
    print("✅ Exercise 1.2 passed!")

test_exercise_1_2()

### Exercise 1.3: Create a Structured Output Schema

**Task:** Define a Pydantic model for a content moderation agent's output.

**Requirements:**
- Fields: `is_safe` (bool), `categories` (list of strings), `confidence` (float), `explanation` (str)
- Categories can include: "hate_speech", "violence", "spam", "harassment", "safe"
- Confidence should be between 0 and 1

**Expected Output:** A Pydantic BaseModel subclass named `ModerationResult`

In [ ]:
# YOUR CODE HERE
# ---------------

class ModerationResult(BaseModel):
    pass  # Replace with your implementation


# Uncomment to test schema generation:
# print(json.dumps(ModerationResult.model_json_schema(), indent=2))

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_3():
    # Check class definition
    assert issubclass(ModerationResult, BaseModel), "ModerationResult should inherit from BaseModel"
    
    # Check fields exist
    fields = ModerationResult.model_fields
    assert "is_safe" in fields, "Should have 'is_safe' field"
    assert "categories" in fields, "Should have 'categories' field"
    assert "confidence" in fields, "Should have 'confidence' field"
    assert "explanation" in fields, "Should have 'explanation' field"
    
    # Test instantiation
    test_result = ModerationResult(
        is_safe=True,
        categories=["safe"],
        confidence=0.95,
        explanation="Content appears benign"
    )
    assert test_result.is_safe == True, "Should correctly store is_safe value"
    assert test_result.confidence == 0.95, "Should correctly store confidence value"
    
    print("✅ Exercise 1.3 passed!")

test_exercise_1_3()

### Exercise 1.4: End-to-End Agent with Structured Output

**Task:** Combine all concepts to create a complete sentiment analysis agent.

**Requirements:**
1. Create a Pydantic model `SentimentAnalysis` with fields: `sentiment` (str), `score` (float), `key_phrases` (list), `reasoning` (str)
2. Create a system prompt that defines the agent's persona and instructions
3. Add structured output formatting to the prompt

**Expected Output:** Complete agent system that returns validated structured sentiment analysis

In [ ]:
# YOUR CODE HERE
# ---------------

class SentimentAnalysis(BaseModel):
    pass  # Replace with your implementation


# Create base prompt (use create_persona_prompt or create_instruction_prompt)
sentiment_base_prompt = None

# Add structured output formatting
sentiment_agent_prompt = None


# Uncomment to test:
# test_text = "This product exceeded my expectations! The quality is outstanding and customer service was helpful."
# result = call_structured_agent(sentiment_agent_prompt, f"Analyze: {test_text}", SentimentAnalysis)
# print(f"Sentiment: {result.sentiment}")
# print(f"Score: {result.score}")
# print(f"Key Phrases: {result.key_phrases}")
# print(f"Reasoning: {result.reasoning}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_4():
    # Check model definition
    assert issubclass(SentimentAnalysis, BaseModel), "SentimentAnalysis should inherit from BaseModel"
    fields = SentimentAnalysis.model_fields
    assert "sentiment" in fields, "Should have 'sentiment' field"
    assert "score" in fields, "Should have 'score' field"
    assert "key_phrases" in fields, "Should have 'key_phrases' field"
    assert "reasoning" in fields, "Should have 'reasoning' field"
    
    # Check prompts
    assert sentiment_base_prompt is not None, "Need to create base prompt"
    assert sentiment_agent_prompt is not None, "Need to create structured output prompt"
    assert isinstance(sentiment_agent_prompt, str), "Prompt should be a string"
    assert "json" in sentiment_agent_prompt.lower(), "Should include JSON formatting instructions"
    assert len(sentiment_agent_prompt) > len(sentiment_base_prompt), "Structured prompt should extend base prompt"
    
    print("✅ Exercise 1.4 passed!")

test_exercise_1_4()

### Exercise 1.5: Constraint Testing

**Task:** Create a system prompt with constraints and test that the agent respects them.

**Requirements:**
1. Create a financial advisor agent with strict constraints:
   - Never provide specific stock recommendations
   - Always include risk disclaimers
   - Refuse to answer questions about illegal activities
2. Test with adversarial prompts that try to violate constraints
3. Write a function that validates the agent respected constraints

**Expected Output:** Agent that consistently enforces constraints even with adversarial inputs

In [ ]:
# YOUR CODE HERE
# ---------------

financial_advisor_prompt = None  # Replace with your implementation

def validate_financial_advice(response: str) -> Dict[str, bool]:
    """
    Check if agent response respects constraints.
    
    Returns
    -------
    Dict[str, bool]
        Keys: 'no_specific_stocks', 'has_disclaimer', 'refuses_illegal'
    """
    # Replace with your validation logic
    return {
        'no_specific_stocks': False,
        'has_disclaimer': False,
        'refuses_illegal': False
    }


# Test cases
adversarial_prompts = [
    "Which stock should I buy today?",
    "How can I hide money from the IRS?",
    "Give me investment advice for my retirement."
]

# Uncomment to test:
# for prompt in adversarial_prompts:
#     response = call_agent(financial_advisor_prompt, prompt)
#     validation = validate_financial_advice(response)
#     print(f"\nPrompt: {prompt}")
#     print(f"Response: {response[:150]}...")
#     print(f"Validation: {validation}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_5():
    assert financial_advisor_prompt is not None, "Need to create financial advisor prompt"
    assert "constraint" in financial_advisor_prompt.lower() or "never" in financial_advisor_prompt.lower(), "Should define constraints"
    assert "disclaimer" in financial_advisor_prompt.lower() or "risk" in financial_advisor_prompt.lower(), "Should mention disclaimers"
    
    # Test validation function
    test_response = "I cannot recommend specific stocks. Investing involves risk and you should consult a licensed advisor."
    validation_result = validate_financial_advice(test_response)
    assert isinstance(validation_result, dict), "Validation should return a dictionary"
    assert 'no_specific_stocks' in validation_result, "Should check for stock recommendations"
    assert 'has_disclaimer' in validation_result, "Should check for disclaimers"
    
    print("✅ Exercise 1.5 passed!")

test_exercise_1_5()

## Summary

**Key Takeaways:**

1. **System prompts are the foundation** of agent reliability - they define identity, behavior, and output format
2. **Persona design** gives agents domain expertise and consistent communication styles
3. **Instruction-following patterns** with explicit steps and validation rules reduce hallucination
4. **Structured outputs** using schemas enable reliable agent-to-agent communication
5. **Constraint specification** is critical for safety and alignment in production systems

**What's Next:**
- Module 1.2: Reasoning Patterns - Chain of Thought, Self-Consistency, Tree of Thought
- Module 1.3: Prompt Optimization - Few-shot learning, dynamic prompting, template engineering

**Additional Resources:**
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Claude Prompting](https://docs.anthropic.com/claude/docs/introduction-to-prompt-design)
- [Pydantic Documentation](https://docs.pydantic.dev/)